学習用，元論文のCNN

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# --- 設定: ファイルパスと学習パラメータ ---
pre = "/home/hiromu/energy"
IMG_ROOT = f'{pre}/data/1201_humomentstest'

# 指定されたCSVファイルパス
TRAIN_CSV_PATH = f'{pre}/src/FF/AFF/dataset_comparison/train_max.csv'
VAL_CSV_PATH   = f'{pre}/src/FF/AFF/dataset_comparison/val_fixed.csv'

# 学習設定
TARGET_EPOCH = 100
SAVE_DIR = f"../save/Org_{TARGET_EPOCH}epoch_fixed_dataset"


# --- 1. CBAM モジュール ---
class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super(CBAM, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels)
        )
        self.sigmoid_channel = nn.Sigmoid()
        self.conv_spatial = nn.Conv2d(2, 1, kernel_size=7, padding=3)
        self.sigmoid_spatial = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(torch.mean(x, dim=(2, 3)))
        max_out = self.fc(torch.amax(x, dim=(2, 3)))
        out = self.sigmoid_channel(avg_out + max_out).unsqueeze(-1).unsqueeze(-1)
        x = x * out
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out = torch.amax(x, dim=1, keepdim=True)
        out = self.sigmoid_spatial(self.conv_spatial(torch.cat([avg_out, max_out], dim=1)))
        return x * out


# --- 2. 標準的な ResBlock (CBAMなし) ---
class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ResBlock, self).__init__()
        self.conv_block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels)
        )
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.conv_block(x)
        return torch.relu(out + identity)


# --- 3. 融合モデル (融合部分にのみCBAMを使用) ---
class FusionModel(nn.Module):
    def __init__(self, num_classes, fusion_strategy='AFF'):
        super(FusionModel, self).__init__()
        self.fusion_strategy = fusion_strategy
        
        self.dl_extractor = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            ResBlock(32, 64),
            ResBlock(64, 64)
        )
        
        self.hc_projector = nn.Sequential(nn.Linear(10, 64), nn.ReLU(inplace=True))
        self.fusion_attention = CBAM(64)
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64, num_classes)
        )

    def forward(self, img, hc_vec):
        Y = self.dl_extractor(img)
        B, C, H, W = Y.shape
        X = self.hc_projector(hc_vec).view(B, C, 1, 1).expand(B, C, H, W)
        weights = self.fusion_attention(X + Y)
        fused = weights * X + (1 - weights) * Y
        return self.classifier(fused)


# --- 4. Datasetクラス ---
class CustomMultiModalDataset(Dataset):
    def __init__(self, dataframe, img_root, scaler=None, transform=None, is_train=True):
        self.df = dataframe.reset_index(drop=True)
        self.img_root = img_root
        self.transform = transform
        self.feature_cols = ['h0', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'size_count', 'R']
        
        features = self.df[self.feature_cols].values
        
        if is_train:
            self.scaler = StandardScaler()
            self.hc_features = self.scaler.fit_transform(features).astype('float32')
        else:
            assert scaler is not None, "Test set requires a fitted scaler from training set."
            self.scaler = scaler
            self.hc_features = self.scaler.transform(features).astype('float32')
            
        self.labels = self.df['Label'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_root, 'mask', self.df.iloc[idx]['category'], self.df.iloc[idx]['filename'])
        
        try:
            image = Image.open(img_path).convert('RGB')
        except (OSError, FileNotFoundError):
            print(f"Error loading: {img_path}")
            image = Image.new('RGB', (256, 256), (0, 0, 0))
            
        if self.transform:
            image = self.transform(image)
            
        hc_feat = torch.tensor(self.hc_features[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, hc_feat, label


# --- 5. ユーティリティ: 学習曲線保存 ---
def save_learning_curve(history, save_path):
    epochs = range(1, len(history['train_loss']) + 1)
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'r-', label='Train Loss')
    plt.title('Training Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['val_acc'], 'b-', label='Val Acc')
    plt.title('Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()
    print(f"Learning curve saved to {save_path}")


# --- 6. メイン学習関数 ---
def train_model():
    BATCH_SIZE = 4
    EPOCHS = TARGET_EPOCH
    LR = 0.0001
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    os.makedirs(SAVE_DIR, exist_ok=True)

    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    print("Loading Datasets from CSV...")
    
    if not os.path.exists(TRAIN_CSV_PATH) or not os.path.exists(VAL_CSV_PATH):
        print(f"Error: CSV files not found.\nCheck paths:\n{TRAIN_CSV_PATH}\n{VAL_CSV_PATH}")
        return

    train_df = pd.read_csv(TRAIN_CSV_PATH)
    val_df = pd.read_csv(VAL_CSV_PATH)
    
    print("-" * 30)
    print(f"Train Data (from {os.path.basename(TRAIN_CSV_PATH)}): {len(train_df)} samples")
    print(f"Val Data   (from {os.path.basename(VAL_CSV_PATH)}):   {len(val_df)} samples")
    print("-" * 30)
    
    train_dataset = CustomMultiModalDataset(train_df, IMG_ROOT, transform=transform, is_train=True)
    val_dataset = CustomMultiModalDataset(val_df, IMG_ROOT, scaler=train_dataset.scaler, transform=transform, is_train=False)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    model = FusionModel(num_classes=3, fusion_strategy='AFF').to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    
    history = {'train_loss': [], 'val_acc': []}
    best_acc = 0.0

    print(f"Starting Training on {DEVICE} for {EPOCHS} epochs...")

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        
        for images, hc_feats, labels in train_loader:
            images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(images, hc_feats)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
        
        epoch_loss = running_loss / len(train_dataset)
        
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, hc_feats, labels in val_loader:
                images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
                outputs = model(images, hc_feats)
                _, preds = torch.max(outputs, 1)
                total += labels.size(0)
                correct += torch.sum(preds == labels.data)
        
        val_acc = (correct.double() / total).item()
        
        history['train_loss'].append(epoch_loss)
        history['val_acc'].append(val_acc)
        
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {epoch_loss:.4f} | Val Acc: {val_acc:.4f}")
        
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), f'{SAVE_DIR}/best_fusion_model.pth')
            print(f"  --> Best Model Saved (Acc: {best_acc:.4f})")
        
        if (epoch + 1) % 5 == 0:
            save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')

    print(f"Training Complete. Best Val Acc: {best_acc:.4f}")
    save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')
    return history

# --- 実行 ---
if __name__ == "__main__":
    history = train_model()

Loading Datasets from CSV...
------------------------------
Train Data (from train_max.csv): 432 samples
Val Data   (from val_fixed.csv):   144 samples
------------------------------
Starting Training on cuda for 100 epochs...
Epoch 1/100 | Loss: 1.1202 | Val Acc: 0.4028
  --> Best Model Saved (Acc: 0.4028)
Epoch 2/100 | Loss: 1.0865 | Val Acc: 0.3472
Epoch 3/100 | Loss: 1.0849 | Val Acc: 0.3750
Epoch 4/100 | Loss: 1.0844 | Val Acc: 0.3750
Epoch 5/100 | Loss: 1.0845 | Val Acc: 0.4028
Learning curve saved to ../save/Org_100epoch_fixed_dataset/learning_curve.png
Epoch 6/100 | Loss: 1.0563 | Val Acc: 0.5139
  --> Best Model Saved (Acc: 0.5139)
Epoch 7/100 | Loss: 1.0399 | Val Acc: 0.4514
Epoch 8/100 | Loss: 1.0005 | Val Acc: 0.5278
  --> Best Model Saved (Acc: 0.5278)
Epoch 9/100 | Loss: 1.0055 | Val Acc: 0.5417
  --> Best Model Saved (Acc: 0.5417)
Epoch 10/100 | Loss: 0.9938 | Val Acc: 0.4306
Learning curve saved to ../save/Org_100epoch_fixed_dataset/learning_curve.png
Epoch 11/100 | Los